# P5 — Evaluación del modelo

## Objetivo
Evaluar el modelo final con métricas complementarias y no únicamente Accuracy.

In [1]:

from pathlib import Path
import sys
import yaml

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "No se encontró config.yaml. "
        "Ejecute Jupyter desde la carpeta del proyecto o notebooks/."
    )

ROOT = find_project_root()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

cfg = yaml.safe_load(
    (ROOT / "config.yaml").read_text(encoding="utf-8")
)

print("Proyecto:", ROOT)


Proyecto: C:\Users\Nicolas Manco\Downloads\CONCRETE_DL


## 1. Cargar modelo entrenado

In [2]:
import torch
from src.models import build_resnet50

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = build_resnet50(
    cfg["model"]["head_hidden"],
    cfg["model"]["dropout"],
    pretrained=False
).to(device)

model.load_state_dict(
    torch.load(
        ROOT/"models/resnet50_finetuned.pt",
        map_location=device
    )
)

model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

## 2. Preparar test interno

In [3]:
from src.data import make_loaders

_, loaders = make_loaders(
    ROOT/cfg["data"]["supervised_split"],
    root=ROOT,
    input_size=cfg["model"]["input_size"],
    batch_size=cfg["training"]["batch_size"],
    num_workers=0
)

print(
    "Imágenes de test:",
    len(loaders["test_internal"].dataset)
)

Imágenes de test: 30


## 3. Calcular métricas

In [4]:
from src.training import run_epoch
import torch.nn as nn

metrics, y_true, y_prob = run_epoch(
    model,
    loaders["test_internal"],
    nn.BCEWithLogitsLoss(),
    device,
    None
)

display(metrics)

{'loss': 0.043582035973668096,
 'accuracy': 1.0,
 'precision': 1.0,
 'recall': 1.0,
 'f1': 1.0,
 'roc_auc': 1.0}

## 4. Matriz de confusión

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

OUT = ROOT/"results/metrics"; OUT.mkdir(parents=True, exist_ok=True)
y_pred = (np.asarray(y_prob) >= cfg["model"]["threshold"]).astype(int)
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5,5)); ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.max())
ax.set_xticks([0,1], ["Normal","Anomalía"]); ax.set_yticks([0,1], ["Normal","Anomalía"])
ax.set_xlabel("Predicción"); ax.set_ylabel("Clase real"); ax.set_title("ResNet50 — matriz de confusión")
for i in range(2):
    for j in range(2): ax.text(j,i,int(cm[i,j]),ha="center",va="center")
fig.tight_layout(); fig.savefig(OUT/"resnet50_confusion_matrix.png", dpi=220, bbox_inches="tight")
plt.show(); plt.close(fig)


## 5. ROC-AUC

In [ ]:
import pandas as pd
from sklearn.metrics import roc_curve, roc_auc_score
fpr, tpr, roc_thresholds = roc_curve(y_true, y_prob); auc = roc_auc_score(y_true, y_prob)
roc_df = pd.DataFrame({"fpr":fpr,"tpr":tpr,"threshold":roc_thresholds})
roc_df.to_csv(OUT/"resnet50_roc_curve.csv", index=False)
fig, ax = plt.subplots(figsize=(6,5)); ax.plot(fpr,tpr,label=f"AUC = {auc:.3f}"); ax.plot([0,1],[0,1],"--")
ax.set_title("ResNet50 — curva ROC"); ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.grid(alpha=.2); ax.legend(); fig.tight_layout(); fig.savefig(OUT/"resnet50_roc_curve.png", dpi=220, bbox_inches="tight")
plt.show(); plt.close(fig)


## 6. Precision–Recall

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
precision, recall, pr_thresholds = precision_recall_curve(y_true, y_prob); ap = average_precision_score(y_true, y_prob)
pr_df = pd.DataFrame({"recall":recall,"precision":precision})
pr_df.to_csv(OUT/"resnet50_precision_recall_curve.csv", index=False)
fig, ax = plt.subplots(figsize=(6,5)); ax.plot(recall,precision,label=f"AP = {ap:.3f}")
ax.set_title("ResNet50 — Precision–Recall"); ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.grid(alpha=.2); ax.legend(); fig.tight_layout(); fig.savefig(OUT/"resnet50_precision_recall_curve.png", dpi=220, bbox_inches="tight")
plt.show(); plt.close(fig)


## Comparación con CNN base e incertidumbre estadística


In [ ]:
from src.models import BaselineCNN
from scipy.stats import beta
from math import comb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Predicciones ResNet50 por imagen
paths = list(loaders["test_internal"].dataset.frame["path"].astype(str))
res_df = pd.DataFrame({"path":paths,"y_true":np.asarray(y_true,dtype=int),"prob_anomaly":np.asarray(y_prob,dtype=float),"y_pred":y_pred})
res_df.to_csv(OUT/"resnet50_test_predictions.csv", index=False)

# CNN baseline sobre exactamente el mismo test
base = BaselineCNN(dropout=cfg["model"]["dropout"]).to(device)
base.load_state_dict(torch.load(ROOT/"models/baseline_best.pt", map_location=device)); base.eval()
base_metrics, base_true, base_prob = run_epoch(base, loaders["test_internal"], nn.BCEWithLogitsLoss(), device, None)
base_pred=(np.asarray(base_prob)>=cfg["model"]["threshold"]).astype(int)
base_df=pd.DataFrame({"path":paths,"y_true":np.asarray(base_true,dtype=int),"prob_anomaly":np.asarray(base_prob,dtype=float),"y_pred":base_pred})
base_df.to_csv(OUT/"baseline_test_predictions.csv", index=False)

# Métricas y Clopper-Pearson exacto
def cp_interval(k,n,alpha=.05):
    lo = 0.0 if k==0 else beta.ppf(alpha/2, k, n-k+1)
    hi = 1.0 if k==n else beta.ppf(1-alpha/2, k+1, n-k)
    return float(lo), float(hi)

def metric_row(name, yt, yp, prob):
    k=int((yt==yp).sum()); n=len(yt); lo,hi=cp_interval(k,n)
    return {"model":name,"n":n,"correct":k,"accuracy":accuracy_score(yt,yp),"accuracy_ci95_low":lo,"accuracy_ci95_high":hi,
            "precision":precision_score(yt,yp,zero_division=0),"recall":recall_score(yt,yp,zero_division=0),
            "f1":f1_score(yt,yp,zero_division=0),"roc_auc":roc_auc_score(yt,prob)}

comp=pd.DataFrame([
    metric_row('CNN baseline', np.asarray(base_true,dtype=int), base_pred, np.asarray(base_prob)),
    metric_row('ResNet50', np.asarray(y_true,dtype=int), y_pred, np.asarray(y_prob)),
])
comp.to_csv(OUT/"model_comparison.csv", index=False)
display(comp.round(4))

# McNemar exacto para predicciones pareadas
base_ok=(np.asarray(base_true,dtype=int)==base_pred); res_ok=(np.asarray(y_true,dtype=int)==y_pred)
b=int(np.sum(base_ok & ~res_ok)); c=int(np.sum(~base_ok & res_ok)); n_disc=b+c
if n_disc==0:
    p_mcnemar=1.0
else:
    tail=sum(comb(n_disc,k) for k in range(0,min(b,c)+1))/(2**n_disc)
    p_mcnemar=min(1.0, 2*tail)
stat=pd.DataFrame([{"baseline_correct_resnet_wrong":b,"baseline_wrong_resnet_correct":c,"discordant_pairs":n_disc,"mcnemar_exact_p":p_mcnemar}])
stat.to_csv(OUT/"mcnemar_exact.csv", index=False)

# Matrices combinadas para el artículo
from sklearn.metrics import confusion_matrix
cm_b=confusion_matrix(base_true,base_pred); cm_r=confusion_matrix(y_true,y_pred)
fig,axs=plt.subplots(1,2,figsize=(7.2,3.2))
for ax,mat,title in zip(axs,[cm_b,cm_r],["CNN base","ResNet50"]):
    ax.imshow(mat, cmap="Blues", vmin=0, vmax=max(cm_b.max(), cm_r.max())); ax.set_xticks([0,1],["Normal","Anom."]); ax.set_yticks([0,1],["Normal","Anom."])
    ax.set_title(title); ax.set_xlabel("Predicción"); ax.set_ylabel("Real")
    for i in range(2):
        for j in range(2): ax.text(j,i,int(mat[i,j]),ha='center',va='center')
fig.tight_layout()
import shutil
for ext in ['png','pdf']:
    out=OUT/f'confusion_comparison.{ext}'
    fig.savefig(out,dpi=240 if ext=='png' else None,bbox_inches='tight')
    shutil.copy2(out,ROOT/f'results/paper/fig4_confusion.{ext}')
plt.close(fig)

print(f"ResNet50: {int((y_pred==np.asarray(y_true)).sum())}/{len(y_true)} correctas; IC95% exacto = {comp.loc[comp.model=='ResNet50','accuracy_ci95_low'].iloc[0]:.4f}–{comp.loc[comp.model=='ResNet50','accuracy_ci95_high'].iloc[0]:.4f}")
print(f"McNemar exacto p={p_mcnemar:.4f} con {n_disc} pares discordantes.")
